# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第12章 韻律研究 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch12_worked_example.R`

対象章: 第12章「韻律研究」通しの作例

図12.1: X-JToBIラベルを重ねたF0曲線（模式図）。
  実際の発話の音響測定ではなく、東京方言のアクセント核(H*+L)と
  音韻句境界音調(%L, H-, L%)がF0曲線のどの形状的特徴と対応するかを
  示すための、理想化された模式的な曲線である。

図12.2: Hz尺度と半音(セミトーン)・z得点正規化の比較。
  声域の異なる2名の仮想話者について、同じ形状のF0パターンを
  異なる声域に乗せてシミュレートし、正規化の有無で話者間の差が
  どう変わって見えるかを対比する。いずれも実測データではない。
--- 図12.1: 模式的なF0曲線とX-JToBIラベル -----------------------------

語頭(%L)からアクセント核(H*+L)までは、単調な立ち上がり1本の山だけにする。
(以前の版は「語頭付近の緩やかな立ち上がり」を独立したガウス項として
加算していたため、意図せず2つ目の無ラベルのピークができてしまい、
「図中のすべての形状的特徴がラベルと対応する」という本図の狙いに反していた。
monoH.FC補間で単調な区間だけを作ることで、この問題を避ける。)

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

rise  <- splinefun(c(0, 25, 45), c(130, 150, 168), method = "monoH.FC")
fall  <- splinefun(c(45, 55, 60), c(168, 118, 114), method = "monoH.FC")

t <- seq(0, 100, length.out = 400)
f0_schematic <- ifelse(
  t <= 45, rise(t),
  ifelse(t <= 60, fall(t),
         114 + 10 * exp(-((t - 75) / 10)^2) - 0.02 * (t - 60))  # H-の緩やかな山 + 句末に向けた微下降
)

schematic_df <- tibble(t = t, f0 = f0_schematic)
labels_df <- tibble(
  t     = c(2, 45, 60, 75),
  f0    = c(132, 168, 114, 124),
  label = c("%L", "H*+L", "L", "H-"),
  vjust = c(1.6, -0.6, 1.6, -0.6)
)

p1 <- ggplot(schematic_df, aes(t, f0)) +
  geom_line(linewidth = 1, colour = "#34518A") +
  geom_text(data = labels_df, aes(t, f0, label = label, vjust = vjust),
            size = 4, fontface = "bold", colour = "#A8681F") +
  scale_x_continuous(limits = c(-2, 100)) +
  scale_y_continuous(limits = c(NA, 178)) +
  labs(x = "発話内の相対時間 (%)", y = "F0 (Hz、模式値)",
       caption = "模式図（実測データではない）。scripts/R/ch12_worked_example.R より生成。")
save_fig(p1, "fig12-01-tobi-schematic", chapter = "ch12")

## --- 図12.2: Hz・半音・z得点の比較 --------------------------------------

n_time <- 50
shape <- plogis(seq(-4, 4, length.out = n_time)) - 0.5   # 共通の形状(立ち上がりパターン)

# 話者A: 声域が低い(100-160Hz)、話者B: 声域が高い(190-300Hz)
speakerA <- 130 + shape * 30
speakerB <- 245 + shape * 55

d_hz <- tibble(
  time     = rep(seq(0, 100, length.out = n_time), 2),
  speaker  = rep(c("話者A", "話者B"), each = n_time),
  hz       = c(speakerA, speakerB)
) |>
  group_by(speaker) |>
  mutate(
    semitone = 12 * log2(hz / mean(hz)),
    z_score  = as.numeric(scale(hz))
  ) |>
  ungroup()

d_long <- d_hz |>
  pivot_longer(cols = c(hz, semitone, z_score), names_to = "scale", values_to = "value") |>
  mutate(scale = factor(scale, levels = c("hz", "semitone", "z_score"),
                         labels = c("生のHz", "半音(セミトーン)", "z得点")))

p2 <- ggplot(d_long, aes(time, value, colour = speaker, linetype = speaker)) +
  geom_line(linewidth = 0.9) +
  facet_wrap(~scale, scales = "free_y", nrow = 1) +
  scale_colour_manual(values = c("話者A" = "#34518A", "話者B" = "#A8681F")) +
  scale_linetype_manual(values = c("話者A" = "solid", "話者B" = "22")) +
  labs(x = "発話内の相対時間 (%)", y = NULL, colour = NULL, linetype = NULL,
       caption = "シミュレーションデータ。scripts/R/ch12_worked_example.R より生成。")
save_fig(p2, "fig12-02-normalization-comparison", chapter = "ch12", width = 8.5, height = 3.2)

range_hz       <- diff(range(c(mean(speakerA), mean(speakerB))))
range_semitone <- 12 * log2(mean(speakerB) / mean(speakerA))
cat("話者A平均F0:", round(mean(speakerA), 1), "Hz / 話者B平均F0:", round(mean(speakerB), 1), "Hz\n")
cat("平均F0の話者間差(Hz):", round(range_hz, 1), "Hz\n")
cat("同じ差を半音に換算すると:", round(range_semitone, 2), "semitones\n")
cat("完了。図は assets/figures/ch12/ に保存した。\n")


## `ch12_annotator_agreement.R`

対象章: 第12章「韻律研究」

「ラベリングの信頼性の評価」節の作例。2名のアノテーターが
同じ発話に句境界の強弱(強い境界/弱い境界)をラベル付けしたと想定し、
単純な一致率とCohenのカッパ係数がどう異なるかを、架空のデータで
確認する。境界の大部分が「弱い」に偏っている(不均衡な分布)場合、
単純な一致率が偶然の一致によって高く見えてしまうことを示す。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_tokens <- 300
p_strong <- 0.15   # 真の境界のうち「強い境界」である割合(不均衡な分布)

true_label <- rbinom(n_tokens, 1, p_strong)

# 各アノテーターは真のラベルをある程度正確に判断するが、一定確率で誤る
make_annotator <- function(true, error_rate) {
  flip <- rbinom(length(true), 1, error_rate)
  ifelse(flip == 1, 1 - true, true)
}

annotator_A <- make_annotator(true_label, 0.12)
annotator_B <- make_annotator(true_label, 0.15)

tab <- table(annotator_A, annotator_B)
print(tab)

percent_agreement <- sum(diag(tab)) / sum(tab)

# Cohenのカッパ係数を手計算する(偶然の一致率を差し引く)
p_o <- percent_agreement
marg_A <- rowSums(tab) / sum(tab)
marg_B <- colSums(tab) / sum(tab)
p_e <- sum(marg_A * marg_B)
kappa <- (p_o - p_e) / (1 - p_e)

cat("単純な一致率:", round(percent_agreement * 100, 1), "%\n")
cat("偶然の一致率(期待値):", round(p_e * 100, 1), "%\n")
cat("Cohenのカッパ係数:", round(kappa, 3), "\n")

# 比較のため、境界がほぼ半々に分布する場合(バランスの取れた分布)も試す
true_label_balanced <- rbinom(n_tokens, 1, 0.5)
annotator_A2 <- make_annotator(true_label_balanced, 0.12)
annotator_B2 <- make_annotator(true_label_balanced, 0.15)
tab2 <- table(annotator_A2, annotator_B2)
p_o2 <- sum(diag(tab2)) / sum(tab2)
marg_A2 <- rowSums(tab2) / sum(tab2)
marg_B2 <- colSums(tab2) / sum(tab2)
p_e2 <- sum(marg_A2 * marg_B2)
kappa2 <- (p_o2 - p_e2) / (1 - p_e2)

cat("\n[比較: 境界がほぼ半々に分布する場合]\n")
cat("単純な一致率:", round(p_o2 * 100, 1), "%\n")
cat("偶然の一致率(期待値):", round(p_e2 * 100, 1), "%\n")
cat("Cohenのカッパ係数:", round(kappa2, 3), "\n")


## `ch12_time_normalization.R`

対象章: 第12章「韻律研究」

「F0の測定・正規化・時間正規化」節の作例。話速の異なる2名の話者が、
言語的には同じ相対位置(発話の70%地点)でF0ピークに達するとき、
秒単位の絶対時間で比較すると、ピーク時刻がずれて見えることを確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

speaker_a_duration <- 0.60   # 秒、話速が速い話者
speaker_b_duration <- 1.00   # 秒、話速が遅い話者
relative_peak_position <- 0.70   # 発話の70%地点でピークに達する(言語的には共通)

peak_time_a_sec <- speaker_a_duration * relative_peak_position
peak_time_b_sec <- speaker_b_duration * relative_peak_position

cat("話者A(発話長0.60秒)のピーク時刻(絶対時間): ", peak_time_a_sec, "秒\n")
cat("話者B(発話長1.00秒)のピーク時刻(絶対時間): ", peak_time_b_sec, "秒\n")
cat("絶対時間で見た2話者のピーク時刻差: ", round(peak_time_b_sec - peak_time_a_sec, 2), "秒\n")
cat("相対時間(0-100%)で見た2話者のピーク位置差: ",
    round((peak_time_b_sec / speaker_b_duration - peak_time_a_sec / speaker_a_duration) * 100, 1),
    "%\n")


## `ch12_measurement_dilution.R`

対象章: 第12章「韻律研究」

「境界と卓立の音響的相関」節の作例。境界前長音化が句末音節の韻
(rhyme)にのみ局所的に生じる場合、測定区間を韻だけに絞るか、
単語全体(複数音節)を平均するかで、検出される効果量がどう変わるかを、
架空のデータで確認する。Wightman et al. (1992)の知見を数値で例示する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 100
true_lengthening_ms <- 40   # 境界直前の韻だけに生じる真の伸長効果

# 句末語は2音節(音節1+音節2)からなり、韻の伸長は音節2にのみ生じるとする
syll1_boundary <- rnorm(n_tokens, 120, 15)
syll2_rhyme_boundary <- rnorm(n_tokens, 150, 15) + true_lengthening_ms
syll1_nonboundary <- rnorm(n_tokens, 120, 15)
syll2_rhyme_nonboundary <- rnorm(n_tokens, 150, 15)

# 測定1: 韻だけに絞った測定(音節2のみ)。伸長率は韻自身の持続時間に対する割合。
effect_rhyme_ms <- mean(syll2_rhyme_boundary) - mean(syll2_rhyme_nonboundary)
pct_rhyme_only <- effect_rhyme_ms / mean(syll2_rhyme_nonboundary) * 100

# 測定2: 単語全体(2音節の合計)で平均した測定。伸長率は単語全体の持続時間に対する割合。
word_total_boundary <- syll1_boundary + syll2_rhyme_boundary
word_total_nonboundary <- syll1_nonboundary + syll2_rhyme_nonboundary
effect_word_ms <- mean(word_total_boundary) - mean(word_total_nonboundary)
pct_word_total <- effect_word_ms / mean(word_total_nonboundary) * 100

cat("真の伸長効果(韻のみに生じる):", true_lengthening_ms, "ms\n\n")
cat("韻だけに絞った測定: 絶対差 =", round(effect_rhyme_ms, 1),
    "ms, 伸長率 =", round(pct_rhyme_only, 1), "%\n")
cat("単語全体(2音節合計)で平均した測定: 絶対差 =", round(effect_word_ms, 1),
    "ms, 伸長率 =", round(pct_word_total, 1), "%\n")
cat("\n伸長率で報告する場合、単語全体を測定区間にすると、真の伸長率のおよそ",
    round(pct_word_total / pct_rhyme_only * 100, 0), "%しか検出されない\n")


## `ch12_prominence_indicators.R`

対象章: 第12章「韻律研究」

「境界と卓立の音響的相関」節の作例。プロミネンスの複数の音響的指標
(F0ピーク・持続時間・強度)が、部分的に独立しながらも相関して現れる
という想定のもとで、単一の指標だけで卓立の有無を判定した場合と、
複数指標を組み合わせて判定した場合とで、真の卓立をどれだけ正しく
検出できるかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 200
true_prominent <- rbinom(n_tokens, 1, 0.5)

# 各指標は真の卓立を部分的にしか反映せず、指標ごとに異なるノイズ源を持つ
f0_peak   <- 200 + true_prominent * 30 + rnorm(n_tokens, 0, 25)
duration  <- 120 + true_prominent * 15 + rnorm(n_tokens, 0, 20)
intensity <- 65  + true_prominent * 3  + rnorm(n_tokens, 0, 4)

judge_single <- function(x) as.integer(x > median(x))
judge_f0        <- judge_single(f0_peak)
judge_duration  <- judge_single(duration)
judge_intensity <- judge_single(intensity)

# 複合指標: 3つの指標をz得点化して平均し、その中央値で二値化する
z <- function(x) (x - mean(x)) / sd(x)
composite <- (z(f0_peak) + z(duration) + z(intensity)) / 3
judge_composite <- judge_single(composite)

accuracy <- function(judged) mean(judged == true_prominent)

cat("F0ピークのみでの判定精度:", round(accuracy(judge_f0) * 100, 1), "%\n")
cat("持続時間のみでの判定精度:", round(accuracy(judge_duration) * 100, 1), "%\n")
cat("強度のみでの判定精度:", round(accuracy(judge_intensity) * 100, 1), "%\n")
cat("3指標の複合スコアでの判定精度:", round(accuracy(judge_composite) * 100, 1), "%\n")
